# CBO Proxy Starter

This notebook loads the bundled CBO starter proxy for median adjusted household income after transfers and federal taxes. It also compares the CBO proxy with bundled FRED public comparison series. The CBO series is reproducible from CBO's official Additional Data for Researchers ZIP, but it is still a starter proxy and not publication-ready as the final custom project metric.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [ ]:
import pandas as pd

from economics.charts import plot_multiple_series, plot_time_series
from economics.loaders import CBO_PROXY_VALUE_COL, load_cbo_proxy
from economics.paths import processed_data_path
from economics.series import add_growth_columns
from economics.tables import build_summary_table


In [ ]:
data_path = processed_data_path("cbo_proxy_median_adjusted_income_after_tax_transfer.csv")
cbo = load_cbo_proxy(data_path)
cbo = add_growth_columns(cbo, CBO_PROXY_VALUE_COL)
cbo.head()


In [ ]:
summary = build_summary_table(
    cbo,
    CBO_PROXY_VALUE_COL,
    label="CBO median adjusted income after transfers and federal taxes",
)
summary


In [ ]:
fig, ax = plot_time_series(
    cbo,
    CBO_PROXY_VALUE_COL,
    title="Median adjusted income after transfers and federal taxes",
    ylabel="2022 dollars",
)
fig


In [ ]:
comparison_files = [
    "fred_real_median_personal_income.csv",
    "fred_real_median_household_income.csv",
    "fred_real_disposable_personal_income_per_capita.csv",
]

comparison_series = [
    cbo[["year", CBO_PROXY_VALUE_COL]]
    .rename(columns={CBO_PROXY_VALUE_COL: "value"})
    .assign(series="CBO adjusted income after transfers and taxes")
]
for filename in comparison_files:
    comparison_series.append(
        pd.read_csv(processed_data_path(filename))[["year", "value", "series"]]
    )

public_proxies = pd.concat(comparison_series, ignore_index=True).sort_values(["series", "year"])
public_proxies["index_value"] = public_proxies.groupby("series")["value"].transform(
    lambda values: values / values.iloc[0] * 100
)
public_proxies.head()


In [ ]:
fig, ax = plot_multiple_series(
    public_proxies,
    value_col="index_value",
    title="Public income/resource proxy comparison",
    ylabel="Index, first observation = 100",
)
fig
